# Finance Forecasting Copilot — Walkthrough

This notebook walks through the three building blocks of the project: the forecasting model, the gross-profit scenario engine, and the LangGraph agent that ties them together. The underlying data is synthetic but exposes the kind of seasonality, promo spikes and shock dynamics you would see in real monthly finance data.

In [ ]:
import sys
from pathlib import Path
if '..' not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from fin_copilot.config import load_config
from fin_copilot.data.loader import load

pd.options.display.float_format = '{:,.2f}'.format
cfg = load_config()
df = load()
print(df.shape)
df.head()

## 1. Historical revenue by category and region

In [ ]:
weekly = (df.set_index('date')
          .groupby(['category', 'region'])['revenue']
          .resample('W').sum()
          .reset_index())

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, cat in zip(axes, cfg.data.categories):
    for reg in cfg.data.regions:
        sub = weekly[(weekly['category'] == cat) & (weekly['region'] == reg)]
        ax.plot(sub['date'], sub['revenue'], label=reg, linewidth=1)
    ax.set_title(cat)
    ax.legend(loc='upper left')
    ax.grid(alpha=0.2)
plt.tight_layout();

## 2. Forecast with prediction intervals

We fit a LightGBM quantile forecaster at q10/q50/q90 and roll it out recursively over a 28-day horizon.

In [ ]:
from fin_copilot.forecasting.model import QuantileForecaster

train = df[df['date'] <= '2024-12-03']
model = QuantileForecaster(quantiles=cfg.forecast.quantiles, params=cfg.forecast.lgbm).fit(train)
forecast = model.predict_recursive(train, horizon_days=28)
forecast.head()

In [ ]:
pair = forecast[(forecast['category'] == 'Footwear') & (forecast['region'] == 'EMEA')].sort_values('date')
hist = train[(train['category'] == 'Footwear') & (train['region'] == 'EMEA')].sort_values('date').tail(90)
actual = df[(df['category'] == 'Footwear') & (df['region'] == 'EMEA') & (df['date'] > '2024-12-03')].sort_values('date')

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hist['date'], hist['revenue'], color='#333', label='history', linewidth=1)
ax.plot(actual['date'], actual['revenue'], color='#e67e22', label='actual (held out)', linewidth=1.3)
ax.plot(pair['date'], pair['point'], color='#2980b9', label='forecast (q50)', linewidth=1.3)
ax.fill_between(pair['date'], pair['q10'], pair['q90'], alpha=0.2, color='#2980b9', label='80% PI')
ax.set_title('Footwear · EMEA · 28-day forecast')
ax.legend();
ax.grid(alpha=0.2);
plt.tight_layout();

## 3. Walk-forward backtest

Six folds, each a 28-day holdout, sliding forward through the final six months.

In [ ]:
from fin_copilot.forecasting.backtest import walk_forward
from fin_copilot.evaluation.forecast_metrics import summary_table, per_series_summary

result = walk_forward(df, cfg.forecast, cfg.backtest.n_folds, cfg.backtest.fold_horizon_days)
print(summary_table(result))
per_series_summary(result)

## 4. Gross-profit scenarios

Starting from the point forecast, we project how a price change and/or cost change would flow through to gross profit, factoring in demand response via a simple elasticity.

In [ ]:
from fin_copilot.scenarios.gross_profit import ScenarioParams, run_scenario

pred_full = model.predict_recursive(df, horizon_days=28)
scenarios = {
    'baseline': ScenarioParams(),
    '+5% price / -1.2 elasticity': ScenarioParams(price_change_pct=0.05, price_elasticity=-1.2),
    '+8% COGS': ScenarioParams(cogs_change_pct=0.08),
    '+3% price / +5% COGS / -0.8 elasticity': ScenarioParams(price_change_pct=0.03, cogs_change_pct=0.05, price_elasticity=-0.8),
}
rows = []
for name, params in scenarios.items():
    r = run_scenario(df, pred_full, params)
    s = r.summary()
    rows.append({'scenario': name, **s})
pd.DataFrame(rows).set_index('scenario')[['scenario_revenue','scenario_gross_profit','delta_revenue_pct','delta_gross_profit_pct']]

## 5. Agent (requires an API key)

The code below only runs if `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` is set.

In [ ]:
import os
if os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY'):
    from fin_copilot.agent.graph import build_agent
    agent = build_agent()
    out = agent.invoke('Forecast Footwear revenue in EMEA over the next 14 days and give me the total.')
    print(out['answer'])
    for tc in out['tool_calls']:
        print('→', tc['name'], tc['args'])
else:
    print('Set OPENAI_API_KEY or ANTHROPIC_API_KEY to run the agent.')